<a href="https://colab.research.google.com/github/kalpesh5536/Sentiment-Analysis-using-NLP-Pipeline-ML-Models-/blob/main/Twitter_Sentiment_Analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# Basic libraries
import pandas as pd   # for data handling
import numpy as np    # for numerical work

# NLP libraries
import re             # for text cleaning (regex)
import nltk           # for NLP tasks

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [32]:
# Download stopwords (run only once)
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
# Load your dataset file
df = pd.read_csv("twitter_training.csv")

# Show data
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [35]:
print('Shape =>',df.shape)

Shape => (74681, 4)


In [36]:
# Rename columns (important)
df.columns = ['id', 'entity', 'sentiment', 'text']
df.head()

,id,entity,sentiment,text
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [37]:
# Keep only needed columns
df = df[['sentiment', 'text']]

df.head()

,sentiment,text
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...


In [39]:
# Remove missing values
df = df.dropna()# Keep only needed columns
df = df[['sentiment', 'text']]

# Remove missing values
df = df.dropna()

In [40]:
print('Shape =>',df.shape)  # to check just

Shape => (73995, 2)


In [41]:
print("Total Rows:", len(df))
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

# Example tweet
print("\nSample Text:")
print(df['text'].iloc[0])

Total Rows: 73995

Class Distribution:
sentiment
Negative      22358
Positive      20654
Neutral       18108
Irrelevant    12875
Name: count, dtype: int64

Sample Text:
I am coming to the borders and I will kill you all,


In [43]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()  # lowercase

    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)        # remove special characters

    words = text.split()  # tokenize

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

In [44]:
df['clean_text'] = df['text'].apply(preprocess_text)

# Check before vs after
df[['text', 'clean_text']].head()

,text,clean_text
0,I am coming to the borders and I will kill you...,come border kill
1,im getting on borderlands and i will kill you ...,im get borderland kill
2,im coming on borderlands and i will murder you...,im come borderland murder
3,im getting on borderlands 2 and i will murder ...,im get borderland murder
4,im getting into borderlands and i can murder y...,im get borderland murder


In [60]:
# Bag of word
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

In [46]:
# TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_text'])
y = df['sentiment']

In [47]:
# Convert labels to numbers (positive,negative to 0,1)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)  # see mapping

['Irrelevant' 'Negative' 'Neutral' 'Positive']


In [49]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [52]:
# Logistic Regression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [53]:
# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [54]:
# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [58]:
# Evaluation Function
def evaluate(y_test, y_pred):
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))
    print("-" * 40)

In [59]:
# Evaluate Models
print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("Naive Bayes")
evaluate(y_test, y_pred_nb)

print("Decision Tree")
evaluate(y_test, y_pred_dt)

Logistic Regression
Accuracy : 0.6878167443746199
Precision: 0.6881333577695401
Recall   : 0.6878167443746199
F1 Score : 0.6857907353094543
----------------------------------------
Naive Bayes
Accuracy : 0.6403811068315427
Precision: 0.655256817190224
Recall   : 0.6403811068315427
F1 Score : 0.6291092024170218
----------------------------------------
Decision Tree
Accuracy : 0.7844448949253328
Precision: 0.7872089480547884
Recall   : 0.7844448949253328
F1 Score : 0.784644131596364
----------------------------------------


From the results, Decision Tree performed best with highest accuracy and F1 score.
This indicates it was able to capture patterns effectively in this dataset.

Logistic Regression performed moderately well, while Naive Bayes had the lowest performance.

However, Decision Trees can overfit in many cases, so Logistic Regression is generally more stable for text data.

TF-IDF performed better because it highlights important words and reduces noise.